# 🔥 Early Fire Detection System — Training Notebook

**Hệ thống Phát hiện Cháy Sớm** using **RT-DETR-L** with SAHI integration.

**Authors:** Vo Xuan Quang (523H0173) — TDTU  
**Model:** RT-DETR-L (Real-Time Detection Transformer, Large)  
**Classes:** Fire (Lửa), Smoke (Khói)

---
## Table of Contents
1. [Setup & Installation](#setup)
2. [Data Preparation](#data-prep)
3. [Data Augmentation Preview](#augmentation)
4. [Model Configuration](#config)
5. [Stage 1: Baseline Training](#stage1)
6. [Stage 2: Hard Negative Mining](#stage2)
7. [Stage 3: SAHI Small Object Training](#stage3)
8. [Evaluation](#evaluation)
9. [Visualization](#visualization)
10. [Export & Deploy](#export)

<a id='setup'></a>
## 1. Setup & Installation

In [ ]:
# Install all required packages
!pip install -q -r requirements.txt

# Verify GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Standard imports
import os
import sys
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Add project root to path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s'
)
logger = logging.getLogger('FireDetection')
logger.info('Setup complete.')

<a id='data-prep'></a>
## 2. Data Preparation

Load from all 5 data folders, validate annotations, show sample images.

In [ ]:
from src.config import load_config
from src.data.preprocessing import validate_annotations, deduplicate_dataset

config = load_config('configs/default.yaml')

DATA_DIRS = config.data.data_dirs
print('Data directories:', DATA_DIRS)

# Validate annotations for all folders
for folder in DATA_DIRS:
    if Path(folder).exists():
        valid, invalid, errors = validate_annotations(folder)
        print(f'\n{folder}:')
        print(f'  Valid annotations: {valid}')
        print(f'  Invalid annotations: {invalid}')
        if errors:
            print(f'  First error: {errors[0]}')
    else:
        print(f'\n{folder}: NOT FOUND — please set up the dataset first.')

In [ ]:
from torch.utils.data import DataLoader
from src.data.dataset import FireSmokeDataset, collate_fn

# Load training dataset (using available folders only)
available_dirs = [d for d in DATA_DIRS if Path(d).exists()]
print(f'Available data directories: {len(available_dirs)}/{len(DATA_DIRS)}')

if available_dirs:
    train_dataset = FireSmokeDataset(
        data_dirs=available_dirs,
        img_size=config.model.img_size,
        split='train',
        train_ratio=config.data.train_split,
    )
    val_dataset = FireSmokeDataset(
        data_dirs=available_dirs,
        img_size=config.model.img_size,
        split='val',
        train_ratio=config.data.train_split,
    )
    print(f'Train samples: {len(train_dataset)}')
    print(f'Val samples:   {len(val_dataset)}')
else:
    print('No data available. Please set up the dataset first (see data/README.md).')

In [ ]:
# Show sample images from the dataset
if available_dirs and len(train_dataset) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('Sample Training Images', fontsize=14)

    for i, ax in enumerate(axes.flatten()):
        if i >= len(train_dataset):
            ax.axis('off')
            continue
        img_tensor, bboxes, labels = train_dataset[i]
        img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)

        ax.imshow(img_np)
        ax.set_title(f'Labels: {labels.tolist()}', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('logs/sample_images.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved to logs/sample_images.png')

<a id='augmentation'></a>
## 3. Data Augmentation Preview

In [ ]:
from src.data.augmentation import get_train_transforms, get_val_transforms, apply_mixup

train_tf = get_train_transforms(img_size=config.model.img_size)
val_tf   = get_val_transforms(img_size=config.model.img_size)

print('Train transforms:', train_tf)
print('\nVal transforms:', val_tf)

In [ ]:
# Visualise augmented samples for a single image
if available_dirs and len(train_dataset) > 0:
    # Load raw image (no transform)
    raw_path = train_dataset.image_paths[0]
    raw_img = np.array(Image.open(raw_path).convert('RGB').resize(
        (config.model.img_size, config.model.img_size)
    ))

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('Augmentation Samples (same source image)', fontsize=13)

    axes[0][0].imshow(raw_img)
    axes[0][0].set_title('Original', fontsize=9)
    axes[0][0].axis('off')

    for i, ax in enumerate(axes.flatten()[1:], start=1):
        result = train_tf(image=raw_img, bboxes=[], class_labels=[])
        aug_img = result['image']
        # Denormalize for display
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        display = ((aug_img * std + mean) * 255).clip(0, 255).astype(np.uint8)
        ax.imshow(display)
        ax.set_title(f'Aug #{i}', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('logs/augmentation_preview.png', dpi=100, bbox_inches='tight')
    plt.show()

<a id='config'></a>
## 4. Model Configuration

In [ ]:
import pprint

config = load_config('configs/default.yaml')
pprint.pprint(config.to_dict())

In [ ]:
from src.models.rtdetr_model import FireDetectionModel, load_model

# Initialise RT-DETR-L model
model = load_model(config)
print('Model architecture:', config.model.architecture)
print('Number of classes:', config.model.num_classes)
print('Class names:', config.model.class_names)
print('Image size:', config.model.img_size)

<a id='stage1'></a>
## 5. Stage 1: Baseline Training

Train on `01_Positive_Standard` and `02_Alley_Context` to establish baseline fire/smoke detection.

In [ ]:
from src.engine.trainer import Trainer

trainer = Trainer(model, config)

# Run Stage 1 (uncomment when dataset is ready)
# trainer.run_baseline_training()

print('Stage 1 trainer ready.')
print('To start training, uncomment: trainer.run_baseline_training()')

<a id='stage2'></a>
## 6. Stage 2: Hard Negative Mining

Retrain adding `03_Negative_Hard_Samples` (LED lights, phở steam, etc.) to reduce false positives.

In [ ]:
# Run Stage 2 (after Stage 1 completes)
# trainer.run_hard_negative_mining()

print('Stage 2 trainer ready.')
print('To start, uncomment: trainer.run_hard_negative_mining()')

<a id='stage3'></a>
## 7. Stage 3: SAHI Small Object Training

Fine-tune with `04_SAHI_Small_Objects` and `05_Real_Situation` for small/distant detection.

In [ ]:
# Run Stage 3 (after Stage 2 completes)
# trainer.run_sahi_finetuning()

print('Stage 3 trainer ready.')
print('To start, uncomment: trainer.run_sahi_finetuning()')

<a id='evaluation'></a>
## 8. Evaluation

Compute mAP@50, mAP@50-95, confusion matrix, FPS benchmark.

In [ ]:
from src.engine.evaluator import (
    compute_map,
    compute_map_95,
    plot_confusion_matrix,
    measure_fps,
    generate_evaluation_report,
)

CLASS_NAMES = config.model.class_names
print('Evaluation utilities loaded.')
print('Class names:', CLASS_NAMES)

In [ ]:
# Example: Run evaluation on validation set
# (Replace with actual predictions and ground truth from your val set)

# Dummy example for demonstration:
sample_predictions = [
    {'class_id': 0, 'confidence': 0.92, 'bbox': [100, 100, 300, 300]},
    {'class_id': 1, 'confidence': 0.78, 'bbox': [200, 50,  450, 200]},
]
sample_ground_truth = [
    {'class_id': 0, 'bbox': [105, 98,  295, 305]},
    {'class_id': 1, 'bbox': [195, 48,  455, 198]},
]

map50   = compute_map(sample_predictions, sample_ground_truth, iou_threshold=0.5)
map50_95 = compute_map_95(sample_predictions, sample_ground_truth)

print(f'mAP@50:    {map50:.4f}')
print(f'mAP@50-95: {map50_95:.4f}')

In [ ]:
# Plot and save confusion matrix
Path('logs').mkdir(exist_ok=True)

plot_confusion_matrix(
    predictions=sample_predictions,
    ground_truth=sample_ground_truth,
    class_names=CLASS_NAMES,
    save_path='logs/confusion_matrix.png',
)

from IPython.display import Image as IPImage
IPImage('logs/confusion_matrix.png', width=500)

In [ ]:
# Generate evaluation report (JSON + CSV)
metrics = {
    'mAP50': round(map50, 4),
    'mAP50_95': round(map50_95, 4),
    'fps': None,  # Fill after FPS benchmark below
    'stage': 'sahi_finetune',
    'model': config.model.architecture,
}

generate_evaluation_report(metrics, save_path='logs/evaluation_report')
print('Report saved to logs/evaluation_report.json and logs/evaluation_report.csv')

<a id='visualization'></a>
## 9. Visualization

Detection overlays, training curves, SAHI slice visualisation.

In [ ]:
from src.utils.visualization import (
    draw_detections,
    plot_training_curves,
    visualize_sahi_slices,
    create_comparison_grid,
)
print('Visualization utilities loaded.')

In [ ]:
# Visualise SAHI slicing on a sample image
if available_dirs and len(train_dataset) > 0:
    sample_img_path = str(train_dataset.image_paths[0])
    img_np = np.array(Image.open(sample_img_path).convert('RGB'))

    # Generate sample SAHI slice coordinates
    h, w = img_np.shape[:2]
    sh, sw = config.sahi.slice_height, config.sahi.slice_width
    oh = int(sh * config.sahi.overlap_height_ratio)
    ow = int(sw * config.sahi.overlap_width_ratio)

    slices = []
    y = 0
    while y < h:
        x = 0
        while x < w:
            slices.append((x, y, min(x + sw, w), min(y + sh, h)))
            x += sw - ow
        y += sh - oh

    visualize_sahi_slices(img_np, slices, 'logs/sahi_slices.png')
    from IPython.display import Image as IPImage
    IPImage('logs/sahi_slices.png', width=700)

In [ ]:
# Plot training curves (requires runs/*/results.csv from Ultralytics)
results_csvs = list(Path('runs').rglob('results.csv'))
if results_csvs:
    plot_training_curves(str(results_csvs[-1]), 'logs/training_curves.png')
    from IPython.display import Image as IPImage
    IPImage('logs/training_curves.png', width=800)
else:
    print('No results.csv found yet. Run training first.')

<a id='export'></a>
## 10. Export & Deploy

Save the best model weights and test the web API.

In [ ]:
# Find and copy best weights
import shutil

best_weights = sorted(Path('runs').rglob('best.pt'))
if best_weights:
    latest = best_weights[-1]
    dest = Path('checkpoints/best_final.pt')
    dest.parent.mkdir(exist_ok=True)
    shutil.copy2(latest, dest)
    print(f'Best weights saved to: {dest}')
    print(f'Source: {latest}')
else:
    print('No trained weights found. Please complete training first.')

In [ ]:
# Test inference via web API
# Make sure the server is running: uvicorn web.main:app --port 8000

import requests
import json

API_BASE = 'http://localhost:8000'

# Health check
try:
    resp = requests.get(f'{API_BASE}/health', timeout=3)
    print('Health check:', json.dumps(resp.json(), indent=2))
except Exception as e:
    print(f'Server not running: {e}')
    print('Start with: uvicorn web.main:app --port 8000')

In [ ]:
# Test prediction on a sample image
if available_dirs and len(train_dataset) > 0:
    sample_path = str(train_dataset.image_paths[0])
    try:
        with open(sample_path, 'rb') as f:
            resp = requests.post(
                f'{API_BASE}/v1/predict',
                files={'file': f},
                timeout=30,
            )
        print(f'Prediction result for {Path(sample_path).name}:')
        print(json.dumps(resp.json(), indent=2))
    except Exception as e:
        print(f'Prediction failed: {e}')